# 26 · Bổ sung `age` + `gender` vào `cohort_study_level.parquet`

`25_ehr_teacher.ipynb` in cảnh báo:

```
demographics tim thay: KHONG CO trong cohort
  ! §2.3 liet ke demographics cho Q_CTX nhung cohort khong co cot nao.
```

Thiết kế của nb20 **đã dành sẵn chỗ** cho demographics trong `Q_CTX` (khối `DEMO`, `tier='ctx'`),
chỉ là cohort thiếu cột. Tuổi và giới là **bối cảnh thuần tuý**, không phải kết quả lâm sàng, nên
chúng thuộc CTX một cách hợp lệ — và mạnh đúng ở các nhãn đang yếu (Cardiomegaly,
Enlarged_Cardiomediastinum, Fracture, Lung_Lesion).

**Ghi thẳng vào file chính** theo yêu cầu. An toàn được bảo đảm bằng 4 lớp:

1. **Backup** có timestamp trước khi ghi bất cứ thứ gì
2. **Chỉ THÊM cột** — không sửa, không xoá, không đổi thứ tự dòng
3. Ghi ra **đĩa local** và kiểm chứng ở đó trước, rồi mới copy đè lên Drive
   (Drive FUSE không đổi tên nguyên tử — `os.replace` để lại cache lệch)
4. **Kiểm chứng sau ghi**: đọc lại và so từng cột cũ với bản trong RAM, phải **bit-for-bit**

Chạy lại nhiều lần vô hại — C1 phát hiện cột đã có thì dừng sớm.

**CPU + BigQuery, ~2 phút.**

## C0 · Config

In [ ]:
import os, json, time, hashlib, shutil
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR = os.path.join(DATA_DIR, 'cohort_v2')
TARGET  = os.path.join(OUT_DIR, 'cohort_study_level.parquet')     # FILE CHINH — se duoc THEM COT
BASE    = os.path.join(OUT_DIR, 'cohort_base.parquet')            # nguon index_time, CHI DOC

PROJECT = 'mimic-ehr-v310'
HOSP    = 'physionet-data.mimiciv_3_1_hosp'                       # y nb01 C2

NEW_COLS = ['age', 'gender']      # dung 2 cot: nb20/nb25 do DEMO theo
                                  # ('anchor_age','age','gender','sex') va lay TAT CA cot khop
                                  # -> them dung 2 de DEMO sach, khong trung lap thong tin
AGE_CAP = 91                      # MIMIC-IV gop moi tuoi >= 89 thanh 91 khi khu dinh danh

assert os.path.exists(TARGET), f'khong thay {TARGET}'
print(f'file dich : {TARGET}')
print(f'  kich thuoc {os.path.getsize(TARGET)/1e6:.1f} MB')
print(f'BigQuery  : {PROJECT} | {HOSP}.patients')
print(f'se them   : {NEW_COLS}')

## C1 · Vân tay bản gốc + kiểm tra idempotent

Ghi lại trạng thái trước khi động vào: số dòng, danh sách cột, và **hash nội dung từng cột**.
Đây là căn cứ để C5 chứng minh không có gì bị hỏng.

In [ ]:
ORIG = pd.read_parquet(TARGET)
N0, COLS0 = len(ORIG), list(ORIG.columns)

def col_sha(s):
    '''Bam noi dung mot cot. Chiu duoc ca cot RAGGED (t1_org_ids, t3_org_ids...) —
    hash_pandas_object khong bam duoc numpy.ndarray nen phai co nhanh du phong.'''
    try:
        return hashlib.sha256(
            pd.util.hash_pandas_object(s, index=False).values.tobytes()).hexdigest()
    except TypeError:
        h = hashlib.sha256()
        for v in s.to_numpy():
            if v is None:
                h.update(b'<NA>')
            elif np.ndim(v) == 0:
                h.update(b'<NA>' if pd.isna(v) else repr(v).encode())
            else:
                a = np.asarray(v)
                h.update(str(a.dtype).encode()); h.update(a.tobytes())
            h.update(b'|')
        return h.hexdigest()

RAGGED = [c for c in COLS0 if ORIG[c].dtype == object
          and ORIG[c].map(lambda v: np.ndim(v) > 0).any()]
SHA0 = {c: col_sha(ORIG[c]) for c in COLS0}
print(f'ban goc: {N0:,} dong x {len(COLS0)} cot')
if RAGGED: print(f'  cot ragged (mang moi o): {RAGGED}')
assert ORIG.study_id.is_unique, 'study_id khong duy nhat -> DUNG'
assert not ORIG.study_id.isna().any()

_have = [c for c in NEW_COLS if c in COLS0]
if _have:
    print(f'\n!! da co san {_have} -> KHONG lam gi them. Dung tai day.')
    ALREADY = True
else:
    ALREADY = False
    print(f'chua co {NEW_COLS} -> tiep tuc')
print(f'\nvan tay 5 cot dau: ' + ' | '.join(f'{c}:{SHA0[c][:8]}' for c in COLS0[:5]))

## C2 · Xác thực BigQuery — **device flow của `gcloud`**

`google.colab.auth.authenticate_user()` không dùng được với tài khoản này. Dùng device flow:
chạy cell → mở link hiện ra → lấy mã → **dán ngược vào ô nhập** ngay dưới cell.

Phải làm lại **mỗi session Colab** (credential không tồn tại qua lần khởi động lại).

In [ ]:
!gcloud auth application-default login --no-launch-browser

In [ ]:
# gan project cho ADC roi thu mot query nho de chac chan da thong
!gcloud config set project {PROJECT}
!gcloud auth application-default set-quota-project {PROJECT} 2>/dev/null || true

from google.cloud import bigquery
_cli = bigquery.Client(project=PROJECT, location='US')
print('BigQuery OK ->', [dict(r) for r in _cli.query('SELECT 1 AS ok').result()])

## C3 · Truy vấn `patients` + `index_time`

In [ ]:
if not ALREADY:
    from google.cloud import bigquery
    q = f'SELECT subject_id, gender, anchor_age, anchor_year FROM `{HOSP}.patients`'
    pat = _cli.query(q).to_dataframe()
    pat = pat.drop_duplicates('subject_id')
    print(f'patients: {len(pat):,} benh nhan')
    print(f'  gender: {dict(pat.gender.value_counts())}')
    print(f'  anchor_age: min {pat.anchor_age.min()} | max {pat.anchor_age.max()} '
          f'| n(=={AGE_CAP}) {int((pat.anchor_age == AGE_CAP).sum()):,}')

    # index_time lay tu cohort_base (cung nguon nb16/nb23 dung)
    tb = (pd.read_parquet(BASE, columns=['study_id', 'index_time'])
          .drop_duplicates('study_id'))
    tb['index_time'] = pd.to_datetime(tb['index_time'])
    print(f'\ncohort_base: {len(tb):,} study | index_time thieu {int(tb.index_time.isna().sum())}')

## C4 · Tính `age` tại thời điểm chụp + kiểm tra tính hợp lý

`anchor_age` là tuổi tại `anchor_year`, **không phải** tại lúc chụp. Công thức:

```
age = anchor_age + (year(index_time) - anchor_year)
```

MIMIC-IV gộp mọi tuổi ≥ 89 thành **91**, nên cột này **bị chặn trên** — không được coi là tuổi
thật ở nhóm già.

In [ ]:
if not ALREADY:
    d = (ORIG[['study_id', 'subject_id']]
         .merge(tb, on='study_id', how='left')
         .merge(pat, on='subject_id', how='left'))
    assert len(d) == N0 and (d.study_id.to_numpy() == ORIG.study_id.to_numpy()).all(), \
        'merge lam doi so dong hoac thu tu -> DUNG'

    miss_pat = int(d.anchor_age.isna().sum()); miss_t = int(d.index_time.isna().sum())
    print(f'khong khop patients : {miss_pat:,} / {N0:,} ({miss_pat/N0:.3%})')
    print(f'thieu index_time    : {miss_t:,} / {N0:,} ({miss_t/N0:.3%})')

    yr = d.index_time.dt.year
    AGE = (d.anchor_age + (yr - d.anchor_year)).astype('float32')
    AGE[d.anchor_age.isna() | d.index_time.isna()] = np.nan
    SEX = d.gender.astype('string')

    ok = AGE.notna()
    print(f'\nage tinh duoc: {int(ok.sum()):,} / {N0:,} ({ok.mean():.2%})')
    print(f'  min {AGE[ok].min():.0f} | p1 {AGE[ok].quantile(.01):.0f} | median {AGE[ok].median():.0f} '
          f'| p99 {AGE[ok].quantile(.99):.0f} | max {AGE[ok].max():.0f}')
    print(f'  n(age >= {AGE_CAP}) = {int((AGE >= AGE_CAP).sum()):,} '
          f'({(AGE >= AGE_CAP).mean():.2%})  <- nhom bi chan tran, KHONG phai tuoi that')
    print(f'  gender: {dict(SEX.value_counts(dropna=False))}')

    assert AGE[ok].min() >= 0, f'co age am: {AGE[ok].min()}'
    assert AGE[ok].max() < 150, f'age phi ly: {AGE[ok].max()}'
    assert ok.mean() > 0.95, f'chi tinh duoc age cho {ok.mean():.1%} study -> xem lai nguon'
    assert set(SEX.dropna().unique()) <= {'M', 'F'}, f'gender la {set(SEX.dropna().unique())}'

    # phan bo theo nhan, chi de xem truoc gia tri du bao
    meta = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json'))); LB = meta['labels']
    tr = ORIG.split == 'train'
    R = [dict(label=l, age_pos=float(AGE[tr & (ORIG[l] == 1)].mean()),
              age_neg=float(AGE[tr & (ORIG[l] == 0)].mean()),
              pct_F_pos=float((SEX[tr & (ORIG[l] == 1)] == 'F').mean())) for l in LB]
    R = pd.DataFrame(R); R['d_age'] = R.age_pos - R.age_neg
    print('\n=== tuoi trung binh theo nhan (TRAIN, nhan U-Ones) ===')
    print(R.reindex(R.d_age.abs().sort_values(ascending=False).index)
          .to_string(index=False, float_format=lambda v: f'{v:.2f}'))

## C5 · **Backup → ghi thêm cột → atomic replace**

Chỉ thêm cột. Không đụng dòng, không đụng cột cũ, không đổi thứ tự.

In [ ]:
if not ALREADY:
    # ---- THU ROUND-TRIP tren mot lat nho TRUOC khi dung vao file that ----
    # neu parquet khong tai tao duoc y nguyen (nhat la cot ragged) thi dung ngay,
    # khong de den luc da ghi de file chinh moi phat hien.
    _probe = ORIG.head(2000).copy()
    _pt = TARGET + '.probe'
    _probe.to_parquet(_pt, index=False)
    _rb = pd.read_parquet(_pt); os.remove(_pt)
    _bad = [c for c in COLS0 if col_sha(_rb[c]) != col_sha(_probe[c])]
    assert not _bad, (f'parquet KHONG round-trip duoc cac cot {_bad} -> DUNG, chua dung '
                      f'vao file chinh. Can xu ly rieng cac cot nay truoc.')
    print(f'round-trip probe: {len(COLS0)}/{len(COLS0)} cot tai tao y nguyen  OK')

    stamp = time.strftime('%Y%m%d_%H%M%S')
    BAK = TARGET.replace('.parquet', f'.backup_{stamp}.parquet')
    shutil.copyfile(TARGET, BAK)
    assert os.path.getsize(BAK) == os.path.getsize(TARGET)
    print(f'BACKUP -> {BAK}  ({os.path.getsize(BAK)/1e6:.1f} MB)')

    NEW = ORIG.copy()
    NEW['age'] = AGE.to_numpy('float32')
    NEW['gender'] = SEX.to_numpy()
    assert len(NEW) == N0
    assert list(NEW.columns) == COLS0 + NEW_COLS, 'cot moi khong nam dung cuoi'

    # ---- KHONG dung os.replace tren Drive ----
    # Drive FUSE khong doi ten nguyen tu; no xoa+tao lai qua API roi de cache inode lech,
    # khien getsize/open bao FileNotFoundError du file DA len Drive.
    # => ghi ra DIA LOCAL truoc, kiem chung o do, roi copy de len Drive (ghi de thuan tuy).
    LOCAL = '/content/_cohort_new.parquet' if os.path.isdir('/content') else TARGET + '.local'
    NEW.to_parquet(LOCAL, index=False)
    _L = pd.read_parquet(LOCAL)
    _bad = [c for c in COLS0 if col_sha(_L[c]) != SHA0[c]]
    _bad += [f'thieu {c}' for c in NEW_COLS if c not in _L.columns]
    assert not _bad and len(_L) == N0, (f'ban LOCAL da sai ({_bad}, {len(_L)} dong) '
                                       f'-> DUNG, file tren Drive CHUA bi dung toi')
    print(f'ban local OK ({os.path.getsize(LOCAL)/1e6:.1f} MB) -> copy len Drive')

    shutil.copyfile(LOCAL, TARGET)                # ghi de truc tiep, khong rename
    for _ in range(10):                           # Drive co do tre dong bo
        time.sleep(1)
        if os.path.exists(TARGET) and os.path.getsize(TARGET) > 0: break
    print(f'GHI    -> {TARGET}  ({os.path.getsize(TARGET)/1e6:.1f} MB, '
          f'{len(COLS0)} -> {len(NEW.columns)} cot)')
    os.remove(LOCAL)

    json.dump(dict(added=NEW_COLS, backup=os.path.basename(BAK), n_rows=int(N0),
                   source=f'{HOSP}.patients', project=PROJECT,
                   formula='age = anchor_age + (year(index_time) - anchor_year)',
                   age_cap_note=f'MIMIC-IV gop tuoi >= 89 thanh {AGE_CAP}; cot age bi chan tren',
                   n_age_missing=int(AGE.isna().sum()), n_age_capped=int((AGE >= AGE_CAP).sum()),
                   timestamp=stamp),
              open(os.path.join(OUT_DIR, 'demographics_provenance.json'), 'w'), indent=1)
    print(f'PROVENANCE -> demographics_provenance.json')

## C6 · Kiểm chứng sau ghi — **cột cũ phải bit-for-bit**

Đọc lại từ đĩa và so với bản trong RAM. Nếu bất kỳ cột cũ nào lệch → phục hồi từ backup ngay.

In [ ]:
if not ALREADY:
    CHK = None
    for _try in range(5):                         # cache Drive co the con lech vai giay
        try:
            CHK = pd.read_parquet(TARGET); break
        except Exception as e:
            print(f'  doc lai lan {_try+1} that bai ({type(e).__name__}), doi 3s...'); time.sleep(3)
    assert CHK is not None, (f'khong doc duoc {TARGET}. File CO THE van tot tren Drive — '
                             f'chay drive.flush_and_unmount() roi mount lai va kiem tay '
                             f'TRUOC KHI phuc hoi tu backup.')
    bad = []
    if len(CHK) != N0: bad.append(f'so dong {len(CHK)} != {N0}')
    for c in COLS0:
        if c not in CHK.columns: bad.append(f'mat cot {c}'); continue
        if col_sha(CHK[c]) != SHA0[c]: bad.append(f'cot {c} DOI NOI DUNG')
    for c in NEW_COLS:
        if c not in CHK.columns: bad.append(f'thieu cot moi {c}')

    if bad:
        print('!! HONG:'); [print('   -', b) for b in bad]
        shutil.copyfile(BAK, TARGET)
        print(f'DA PHUC HOI tu {BAK}')
        raise AssertionError('ghi that bai — dataset da duoc tra ve nguyen trang')

    print(f'{len(COLS0)}/{len(COLS0)} cot cu: bit-for-bit KHONG DOI')
    print(f'{len(NEW_COLS)} cot moi: {NEW_COLS}')
    print(f'so dong: {len(CHK):,} (khong doi)')
    print(f'\nage    : {CHK.age.notna().mean():.2%} co gia tri | median {CHK.age.median():.0f}')
    print(f'gender : {dict(CHK.gender.value_counts(dropna=False))}')
    print(f'\nbackup giu tai: {BAK}')

print(f'''
================================================================
BUOC TIEP THEO — QUAN TRONG

nb25 C2 CACHE p_ehr theo file:
    label_pipeline_v1/_ehr_teacher/p_ehr_ctx.npy
    label_pipeline_v1/_ehr_teacher/p_ehr_full.npy
    label_pipeline_v1/_ehr_teacher/p_ehr_full_shuf.npy

Neu KHONG xoa, chay lai nb25 se doc cache cu va demographics SE KHONG duoc dung.
Xoa ca ba roi chay lai nb25 tu C0:

    import shutil; shutil.rmtree(f'{{DATA_DIR}}/label_pipeline_v1/_ehr_teacher')

Khi do C1b se in `demographics tim thay: ['age', 'gender']` thay vi 'KHONG CO'.
Q_CTX se tu 22 -> 24 cot. Cong E0-a van giu nguyen nguong +0.0100 da khoa.
================================================================''')